In [1]:
#1 ทำ preprocessing สำหรับ log (For Training Stage 2)

def add_prefix_token(text): # log data ต้องผ่าน code นี้ก่อน training / inference
    # clean log
    text = text.replace("\t", " ")
    text = text.strip()
    # add token
    if text[0].isalpha() or text[3].isalpha():
        return "[SQL]\n" + text
    else:
        return "[LOG]\n" + text

In [4]:
#2 โหลด CSV + clean

from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="All Data Stage 3(1).csv"
)["train"]

dataset = dataset.map(
    lambda x: {"text": add_prefix_token(x["query log"])}
)

dataset = dataset.remove_columns(["query log", "status"])
dataset = dataset.rename_column("label", "labels")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1810 [00:00<?, ? examples/s]

In [5]:
#3 ทำ Tokenization
from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained(
#     "google-bert/bert-base-uncased"
# )

LOCAL_MODEL_PATH = "D:/New Finetune Hackathon/Finetuned Bert Model State 2/checkpoint-78"  # หรือ path อื่น

tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_PATH,
    use_fast=True
)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)


Map:   0%|          | 0/1810 [00:00<?, ? examples/s]

In [8]:
tokenizer("hello world")


{'input_ids': [101, 7592, 2088, 102], 'token_type_ids': [0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1]}

In [6]:
#4 Train / Validation Split

dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]


In [7]:
#5 โหลดโมเดลสำหรับ Binary Classification

from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    LOCAL_MODEL_PATH,
    num_labels=2
)


In [8]:
#6 Training Configuration (เหมาะกับ Log)
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="Finetuned Bert Model State 2",
    eval_strategy="epoch", #เลิกใช้ evaluation_strategy แล้ว
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",   # 🔴 ปิด wandb
)


In [9]:
#7 Metric (สำคัญมากสำหรับ Anomaly)

from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [10]:
#8 เริ่ม Fine-tune 🚀

from transformers import Trainer, EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]  #หยุด Train เมื่อค่า F1 ไม่ดีขึ้น
)

trainer.train()


C:\Users\aungl\AppData\Local\Temp\ipykernel_2712\3697966789.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.466900,0.350250,0.856354,0.875000,0.772727,0.820690
2,0.276400,0.321924,0.867403,0.920635,0.753247,0.828571
3,0.196200,0.327877,0.895028,0.858025,0.902597,0.879747
4,0.113800,0.377142,0.886740,0.846626,0.896104,0.870662


TrainOutput(global_step=364, training_loss=0.2503465567971324, metrics={'train_runtime': 3866.6828, 'train_samples_per_second': 1.498, 'train_steps_per_second': 0.094, 'total_flos': 1523939232645120.0, 'train_loss': 0.2503465567971324, 'epoch': 4.0})